In [ ]:
from openai import OpenAI

client = OpenAI()

sys_prompt = """
Bạn là công cụ trích xuất thông tin từ file PDF (hóa đơn/biên lai/chứng từ).
Nhiệm vụ:
- Đọc toàn bộ nội dung PDF
- Sửa lỗi OCR (dấu tiếng Việt, chính tả)
- Ánh xạ dữ liệu vào JSON với các trường sau:
{
    "Số phiếu": ,
    "Ngày chứng từ(mm/dd/yyyy)": ,
    "Ngày xuất(mm/dd/yyyy)": ,
    "Mã FG": ,
    "Đvt": ,
    "Số lượng(Thùng)": ,
    "Số lượng(Pcs)": ,
    "Kho nhận": ,
    "Địa chỉ": ,
    "Mã AR": 
}
Chỉ trả về JSON hợp lệ, không thêm giải thích.
"""

try:
    # 1. Upload file PDF
    uploaded_file = client.files.create(
        file=open("data/DN2305.015_sample.pdf", "rb"),
        purpose="assistants"
    )
    file_id = uploaded_file.id

    # 2. Gọi model xử lý
    resp = client.responses.create(
        model="gpt-4o",
        input=[
            {
                "role": "user",
                "content": [
                    {"type": "input_text", "text": sys_prompt},
                    {"type": "input_file", "file_id": file_id}
                ]
            }
        ]
    )

    print(resp.output_text)

finally:
    # 3. Xóa file trên server để không lưu lại
    if "file_id" in locals():
        client.files.delete(file_id)

```json
{
    "Số phiếu": "2305/015",
    "Ngày chứng từ(mm/dd/yyyy)": "05/18/2023",
    "Ngày xuất(mm/dd/yyyy)": "05/18/2023",
    "Mã FG": "BV25SX",
    "Đvt": "pcs",
    "Số lượng(Thùng)": null,
    "Số lượng(Pcs)": "1",
    "Kho nhận": "CÔNG TY TNHH SEKISUI VIỆT NAM",
    "Địa chỉ": "Ô số 14.14, tòa nhà Cornerstone, số 16 Phan Chu Trinh, Hoàn Kiếm, TP.Hà Nội",
    "Mã AR": null
}
```
